In [4]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from typing import TypedDict,Annotated
from langchain_core.messages import BaseMessage,HumanMessage
from langgraph.prebuilt import  ToolNode,tools_condition
from langchain_core.tools import  tool
import asyncio
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph.message import add_messages


In [2]:
load_dotenv()

True

In [3]:
llm = ChatGoogleGenerativeAI(model = "gemini-2.5-flash")

In [ ]:
#MCP client for Local FastMCP Server
client  = MultiServerMCPClient({
    "expense": {
            "transport": "streamable_http",  # if this fails, try "sse"
            "url": "https://splendid-gold-dingo.fastmcp.app/mcp"
        },
    "arith": {
            "transport": "stdio",
            "command": "python3",          
            "args": ["/Users/Lenevo/Desktop/mcp-math-server/main.py"],
        },
})

In [5]:
#Deining State
class ChatState(TypedDict):
    messages : Annotated[list[BaseMessage],add_messages]

In [7]:
#building graph function
async def build_graph():
    tools = await client.get_tools()
    print(tools)
    
    #binding tools
    llm_with_tools = llm.bind_tools(tools)
    
    #defining nodes
    async def chat_node(state:ChatState):
        messages = state['messages']
        response = await  llm_with_tools.ainvoke(messages)
        return {"messages":[response]}
    
    #defining toolnode
    tool_node = ToolNode(tools)
    
    #defining graph and graph nodes
    graph = StateGraph(ChatState)
    
    graph.add_node("chat_node",chat_node)
    graph.add_node("tools",tool_node)
    
    #defining graph edges
    graph.add_edge(START,"chat_node")
    graph.add_conditional_edges("chat_node",tools_condition)
    graph.add_edge("tools","chat_node")
    
    chatbot = graph.compile()
    return chatbot

In [ ]:
async def main():
    chatbot = await build_graph()
    
    #running the graph
    result = await chatbot.ainvoke({"messages":[HumanMessage(content="Give me all my expenses for the month of Nov from 1 Nov to 30 Nov")]})
    print(result['messages'][-1].content)

In [ ]:
if __name__ == "__main__":
    asyncio.run(main())